In [1]:
import pandas as pd 

df = pd.read_csv("C:/Users/msree/Desktop/marketing-campaign-analysis/data/raw/marketing_campaign_data_messy.csv") 
print(df) 

      Campaign_ID              Campaign_Name           Start_Date    End_Date  \
0        CMP-00001       Q4_Summer_CMP-00001  2023-11-24 00:00:00  2023-12-13   
1        CMP-00002       Q1_Launch_CMP-00002  2023-05-06 00:00:00  2023-05-12   
2        CMP-00003       Q3_Winter_CMP-00003  2023-12-13 00:00:00  2023-12-20   
3        CMP-00004  Q1_BlackFriday_CMP-00004           2023-10-30  2023-11-03   
4        CMP-00005       Q2_Winter_CMP-00005  2023-04-22 00:00:00  2023-04-23   
...            ...                       ...                  ...         ...   
2015     CMP-00400       Q3_Summer_CMP-00400  2023-10-31 00:00:00  2023-11-13   
2016     CMP-01255       Q4_Summer_CMP-01255  2023-09-01 00:00:00  2023-09-26   
2017     CMP-01050       Q2_Launch_CMP-01050  2023-02-09 00:00:00  2023-02-21   
2018     CMP-01118       Q4_Winter_CMP-01118  2023-03-30 00:00:00  2023-04-27   
2019     CMP-01554       Q4_Launch_CMP-01554  2023-06-26 00:00:00  2023-07-09   

         Channel  Impressio

In [4]:
print(df.columns) 

# Clean the header values 
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_') 
print(df.columns) 

Index(['campaign_id', 'campaign_name', 'start_date', 'end_date', 'channel',
       'impressions', 'clicks', 'spend', 'conversions', 'active', 'clicks',
       'campaign_tag'],
      dtype='object')
Index(['campaign_id', 'campaign_name', 'start_date', 'end_date', 'channel',
       'impressions', 'clicks', 'spend', 'conversions', 'active', 'clicks',
       'campaign_tag'],
      dtype='object')


In [ ]:
# Remove duplicate column clicks 
df = df.loc[:, ~df.columns.duplicated()] 
print(df.columns) 

Index(['campaign_id', 'campaign_name', 'start_date', 'end_date', 'channel',
       'impressions', 'clicks', 'spend', 'conversions', 'active',
       'campaign_tag'],
      dtype='object')


In [7]:
# Check the dtype of dates 
print(df.start_date.dtype) 

object


In [ ]:
# Change the dtype of start_date to datetime 
df['start_date'] = pd.to_datetime(df['start_date'], errors='coerce') 
print(df.start_date.dtype) 

datetime64[ns]


In [9]:
print(df.end_date.dtype) 

object


In [ ]:
# Change the dtype of end_date to datetime 
df['end_date'] = pd.to_datetime(df['end_date'], errors='coerce') 
print(df.end_date.dtype) 

datetime64[ns]


In [12]:
df['spend']

0       $102.82
1         24.33
2       1323.39
3       2180.38
4        252.44
         ...   
2015    $503.95
2016     1641.0
2017     883.82
2018     4198.5
2019    1315.59
Name: spend, Length: 2020, dtype: object

In [13]:
# Handle inconsistent values 
print(df.spend.dtype) 

object


In [ ]:
# Find records with inconsistent data in the spend column 
spend_masked = df['spend'].astype(str).str.contains(r'\$') 
print(df.loc[spend_masked, ['campaign_id', 'spend']].head(3)) 

   campaign_id     spend
0    CMP-00001   $102.82
21   CMP-00022   $2428.4
22   CMP-00023  $4726.22


In [ ]:
# Replace the inconsistent characters 
df['spend'] = df['spend'].astype(str).str.replace(r'[^\d.-]', '', regex=True) 
df['spend'] = pd.to_numeric(df['spend']) 

print(df.loc[spend_masked, ['campaign_id', 'spend']].head(3)) 

   campaign_id    spend
0    CMP-00001   102.82
21   CMP-00022  2428.40
22   CMP-00023  4726.22


In [16]:
df

,campaign_id,campaign_name,start_date,end_date,channel,impressions,clicks,spend,conversions,active,campaign_tag
0,CMP-00001,Q4_Summer_CMP-00001,2023-11-24,2023-12-13,TikTok,16795,197,102.82,20.0,Y,TI
1,CMP-00002,Q1_Launch_CMP-00002,2023-05-06,2023-05-12,Facebook,1860,30,24.33,1.0,0,FA
2,CMP-00003,Q3_Winter_CMP-00003,2023-12-13,2023-12-20,Email,77820,843,1323.39,51.0,No,EM
3,CMP-00004,Q1_BlackFriday_CMP-00004,NaT,2023-11-03,TikTok,55886,2019,2180.38,135.0,True,TI
4,CMP-00005,Q2_Winter_CMP-00005,2023-04-22,2023-04-23,Facebook,7265,169,252.44,30.0,Yes,FA
...,...,...,...,...,...,...,...,...,...,...,...
2015,CMP-00400,Q3_Summer_CMP-00400,2023-10-31,2023-11-13,TikTok,30592,586,503.95,77.0,1,TI
2016,CMP-01255,Q4_Summer_CMP-01255,2023-09-01,2023-09-26,Google Ads,20097,897,1641.00,162.0,0,GO
2017,CMP-01050,Q2_Launch_CMP-01050,2023-02-09,2023-02-21,Instagram,33254,1117,883.82,214.0,0,IN
2018,CMP-01118,Q4_Winter_CMP-01118,2023-03-30,2023-04-27,Facebook,68728,2960,4198.50,591.0,Yes,FA


In [ ]:
print(df.channel.unique()) 

['TikTok' 'Facebook' 'Email' 'Instagram' 'Google Ads' 'E-mail' nan 'Gogle'
 'Tik_Tok' 'Facebok' 'Insta_gram']


In [25]:
# Found categorical typos in channel column 
channel_map = {
    'Tik_Tok' : 'TikTok', 
    'Facebok' : 'Facebook', 
    'E-mail' : 'Email', 
    'Insta_gram' : 'Instagram',
    'Gogle' : 'Google Ads' 
}

In [ ]:
# Replace channel column values with the channel_map 
df['channel'] = df['channel'].replace(channel_map) 
print(df.channel.unique()) 

['TikTok' 'Facebook' 'Email' 'Instagram' 'Google Ads' nan]


In [27]:
df

,campaign_id,campaign_name,start_date,end_date,channel,impressions,clicks,spend,conversions,active,campaign_tag
0,CMP-00001,Q4_Summer_CMP-00001,2023-11-24,2023-12-13,TikTok,16795,197,102.82,20.0,Y,TI
1,CMP-00002,Q1_Launch_CMP-00002,2023-05-06,2023-05-12,Facebook,1860,30,24.33,1.0,0,FA
2,CMP-00003,Q3_Winter_CMP-00003,2023-12-13,2023-12-20,Email,77820,843,1323.39,51.0,No,EM
3,CMP-00004,Q1_BlackFriday_CMP-00004,NaT,2023-11-03,TikTok,55886,2019,2180.38,135.0,True,TI
4,CMP-00005,Q2_Winter_CMP-00005,2023-04-22,2023-04-23,Facebook,7265,169,252.44,30.0,Yes,FA
...,...,...,...,...,...,...,...,...,...,...,...
2015,CMP-00400,Q3_Summer_CMP-00400,2023-10-31,2023-11-13,TikTok,30592,586,503.95,77.0,1,TI
2016,CMP-01255,Q4_Summer_CMP-01255,2023-09-01,2023-09-26,Google Ads,20097,897,1641.00,162.0,0,GO
2017,CMP-01050,Q2_Launch_CMP-01050,2023-02-09,2023-02-21,Instagram,33254,1117,883.82,214.0,0,IN
2018,CMP-01118,Q4_Winter_CMP-01118,2023-03-30,2023-04-27,Facebook,68728,2960,4198.50,591.0,Yes,FA


In [28]:
print(df['active'].unique()) 

['Y' '0' 'No' 'True' 'Yes' '1' 'False']


In [29]:
# Handle mixed boolean values 
mixed_map = {
    'Y' : True, 
    'Yes' : True, 
    '1' : True, 
    'No' : False, 
    '0' : False 
}

In [31]:
df['active'] = df['active'].map(mixed_map).fillna(False).astype(bool) 
print(df['active'].unique()) 

[ True False]


C:\Users\msree\AppData\Local\Temp\ipykernel_38832\378513337.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['active'] = df['active'].map(mixed_map).fillna(False).astype(bool)


In [32]:
# Logical Integrity 
impossible_mask = df['clicks'] > df['impressions'] 
print(df.loc[impossible_mask, ['campaign_id', 'spend', 'impressions']].head(3)) 

Empty DataFrame
Columns: [campaign_id, spend, impressions]
Index: []


In [33]:
# Handling local integrity 
time_mask = df['start_date'] > df['end_date'] 
print(df.loc[time_mask, ['campaign_id', 'start_date', 'end_date']].head(3)) 

   campaign_id start_date   end_date
23   CMP-00024 2023-05-06 2023-05-01
54   CMP-00055 2023-09-01 2023-08-27
71   CMP-00072 2023-02-01 2023-01-27


In [35]:
# Found such data inconsistency where end_date is before the start_date 
# Assuming campaigns run for 30 days 
df.loc[time_mask, ['end_date']] = df.loc[time_mask, 'start_date'] + pd.Timedelta(days=30) 
print(df.loc[time_mask, ['campaign_id', 'start_date', 'end_date']].head(3)) 

   campaign_id start_date   end_date
23   CMP-00024 2023-05-06 2023-06-05
54   CMP-00055 2023-09-01 2023-10-01
71   CMP-00072 2023-02-01 2023-03-03


In [36]:
# Handling Outliers 
quantile_1 = df['spend'].quantile(0.25) 
quantile_3 = df['spend'].quantile(0.75) 
IQR = quantile_3 - quantile_1 
upper_limit = quantile_1 + (3 * IQR) 
outlier_mask = df['spend'] > upper_limit 
print(df.loc[outlier_mask, ['campaign_id', 'spend']].head(3)) 
df.loc[outlier_mask, 'spend'] = upper_limit 
print(df.loc[outlier_mask, ['campaign_id', 'spend']].head(3)) 

    campaign_id    spend
119   CMP-00120  7471.52
130   CMP-00131  6688.28
189   CMP-00190  7777.24
    campaign_id      spend
119   CMP-00120  6615.1525
130   CMP-00131  6615.1525
189   CMP-00190  6615.1525


In [39]:
# Feature Extraction 
print(df['campaign_name']) 
print(df.campaign_name.dtype) 

0            Q4_Summer_CMP-00001
1            Q1_Launch_CMP-00002
2            Q3_Winter_CMP-00003
3       Q1_BlackFriday_CMP-00004
4            Q2_Winter_CMP-00005
                  ...           
2015         Q3_Summer_CMP-00400
2016         Q4_Summer_CMP-01255
2017         Q2_Launch_CMP-01050
2018         Q4_Winter_CMP-01118
2019         Q4_Launch_CMP-01554
Name: campaign_name, Length: 2020, dtype: object
object


In [40]:
# Extract the season from the campaign_name 
df['season'] = df['campaign_name'].str.extract(r'Q\d_([^_]+)_') 
print(df[['campaign_name', 'season']].head(3)) 

         campaign_name  season
0  Q4_Summer_CMP-00001  Summer
1  Q1_Launch_CMP-00002  Launch
2  Q3_Winter_CMP-00003  Winter


In [41]:
df.to_csv("marketing_campaign_cleaned.csv", index=False) 